# 2. Train, compare, and evaluate

**Question:** Can customer activity help rank customers who later leave?

Run notebook 01 first. This stage compares a simple logistic regression baseline with gradient boosting.
Only training rows fit parameters; validation selects a winner; the test set is used after selection.
There is no large hyperparameter search. The extra calibration step is retained because later scoring
uses probabilities, not just class labels. Calibration can help; it does not guarantee accurate probabilities.

In [ ]:
import sys
from pathlib import Path

# Jupyter may start in the project root or in notebooks/.
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "data" / "source.json").exists():
    raise FileNotFoundError("Open Jupyter from the RetainAI project folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
OUTPUT = ROOT / "output"
OUTPUT.mkdir(exist_ok=True)

## Load the three tables

Each notebook starts its own Python session. It reads saved tables instead of depending on variables
left in another notebook. If the files are missing, stop and run notebook 01.

In [ ]:
import json
from datetime import datetime, timezone

import joblib
import numpy as np
import pandas as pd
import sklearn

from retainai.features import CATEGORICAL, DERIVED, INPUT_COLUMNS, NUMERIC, add_features

paths = [OUTPUT / f"{name}.csv" for name in ["train", "validation", "test"]]
if not all(path.exists() for path in paths):
    raise FileNotFoundError("Run notebooks/01_data.ipynb first.")
training, validation, test = [pd.read_csv(path) for path in paths]
labels = training["churn"]
print(
    "Training rows:",
    len(training),
    "Validation rows:",
    len(validation),
    "Test rows:",
    len(test),
)

## Define what a good result means

Accuracy alone can look good by predicting that everybody stays. We use average precision (AP) to
compare rankings of the less common churn class. Higher AP and ROC-AUC are better.
Recall at top 20% asks what fraction of churners appear in the highest-risk fifth of customers.
Lift at top 10% compares that group's churn fraction with the whole test table.
Brier score is mean squared probability error: smaller is better; it reflects more than calibration alone.

These top-fraction metrics rank by probability. The later campaign list ranks by probability times value.
They answer different questions. Stable sorting gives repeatable ordering for ties.

In [ ]:
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score


def evaluate(labels, probabilities):
    labels = np.asarray(labels)
    probabilities = np.asarray(probabilities)
    order = np.argsort(-probabilities, kind="stable")
    top_20 = order[: max(1, int(np.ceil(len(labels) * 0.20)))]
    top_10 = order[: max(1, int(np.ceil(len(labels) * 0.10)))]
    return {
        "average_precision": float(average_precision_score(labels, probabilities)),
        "roc_auc": float(roc_auc_score(labels, probabilities)),
        "recall_at_top_20pct": float(labels[top_20].sum() / labels.sum()),
        "lift_at_top_10pct": float(labels[top_10].mean() / labels.mean()),
        "brier_score": float(brier_score_loss(labels, probabilities)),
    }

## Build preprocessing and classifier as one pipeline

The two ratios are added by `retainai/features.py`. Their +1 denominators prevent division by zero;
seconds_per_call is a smoothed ratio, not an exact average. That helper must remain importable when
the saved pipeline is loaded in another Python session.

The column transformer selects inputs explicitly, excluding churn, customer ID, and customer value.
Logistic regression benefits from scaling numeric inputs. One-hot encoding gives each category its
own indicator column. The tree keeps numeric values unscaled. All fitting stays inside the pipeline.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler


def build_pipeline(classifier, scale=False):
    preprocessing = ColumnTransformer(
        [
            (
                "numbers",
                StandardScaler() if scale else "passthrough",
                NUMERIC + DERIVED,
            ),
            (
                "categories",
                OneHotEncoder(handle_unknown="ignore", sparse_output=False),
                CATEGORICAL,
            ),
        ]
    )
    return Pipeline(
        [
            ("features", FunctionTransformer(add_features)),
            ("preprocessing", preprocessing),
            ("classifier", classifier),
        ]
    )

## Create the two candidates

Logistic regression learns a weighted combination of inputs. Boosting combines many small trees
and can model nonlinear patterns. The settings below are fixed for this learning example.

Sigmoid calibration learns an adjustment to tree probabilities using five internal folds.
The folds also keep identical profiles together. Wrapping the whole pipeline makes preprocessing
fit separately inside each fold. This step trains several trees and is the slowest cell here.

In [ ]:
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold

baseline = build_pipeline(
    LogisticRegression(max_iter=1000, random_state=42), scale=True
)
tree = build_pipeline(
    HistGradientBoostingClassifier(
        max_iter=150, learning_rate=0.05, l2_regularization=1.0, random_state=42
    )
)
groups = pd.factorize(pd.MultiIndex.from_frame(training[INPUT_COLUMNS]))[0]
splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
folds = list(splitter.split(training, labels, groups=groups))
assert all(set(groups[fit]).isdisjoint(groups[cal]) for fit, cal in folds)
models = {
    "logistic_regression": baseline,
    "calibrated_gradient_boosting": CalibratedClassifierCV(
        tree, method="sigmoid", cv=folds
    ),
}

## Fit on training rows and choose using validation

`fit` learns from examples. `predict_proba(... )[:, 1]` takes the probability of class 1 (churn).
Choose the highest validation AP; lower Brier score breaks an AP tie. Test results do not choose the winner.

In [ ]:
validation_metrics = {}
for name, candidate in models.items():
    candidate.fit(training, labels)
    validation_metrics[name] = evaluate(
        validation.churn, candidate.predict_proba(validation)[:, 1]
    )
display(pd.DataFrame(validation_metrics).T.round(4))
selected = max(
    validation_metrics,
    key=lambda name: (
        validation_metrics[name]["average_precision"],
        -validation_metrics[name]["brier_score"],
    ),
)
model = models[selected]
print("Selected:", selected)

## Evaluate the chosen model, then check for accidental input leakage

This is the final holdout evaluation for this run. Repeatedly tuning against these test results
would turn the test set into another validation set. Changing ID, value, labels, or precomputed
ratios must not alter predictions: only explicitly selected inputs and recomputed ratios count.

In [ ]:
test_probability = model.predict_proba(test)[:, 1]
test_metrics = evaluate(test.churn, test_probability)
display(pd.Series(test_metrics).round(4))

sample = test.head(20)
tampered = sample.assign(
    churn=1 - sample.churn,
    customer_id="changed",
    customer_value=999999,
    seconds_per_call=-999,
    call_failure_rate=999,
)
np.testing.assert_allclose(model.predict_proba(sample), model.predict_proba(tampered))
print("Passed: labels, IDs, value, and supplied ratios cannot change predictions.")

## Save the fitted model and readable results

Joblib saves the trained pipeline, selection details, and training reference table for drift comparison.
Load only model files you created yourself: this serialization format can execute Python code.
We reload the file and check predictions here. Notebook 03 will also load it in a fresh session.

In [ ]:
metadata = {
    "selected_model": selected,
    "trained_at_utc": datetime.now(timezone.utc).isoformat(),
    "seed": 42,
    "sklearn_version": sklearn.__version__,
    "selection_rule": "Highest validation AP; lower Brier score breaks ties",
    "validation": validation_metrics,
    "test": test_metrics,
    "split_rows": {
        "train": len(training),
        "validation": len(validation),
        "test": len(test),
    },
    "scope": "Learning example: grouped random holdout, not a deployment evaluation",
}
joblib.dump(
    {"model": model, "metadata": metadata, "reference": training},
    OUTPUT / "model.joblib",
)
(OUTPUT / "metrics.json").write_text(
    json.dumps(metadata, indent=2, allow_nan=False), encoding="utf-8"
)
reloaded = joblib.load(OUTPUT / "model.joblib")
np.testing.assert_allclose(
    reloaded["model"].predict_proba(test)[:, 1], test_probability
)
for name in ["scores.csv", "drift.json"]:
    (OUTPUT / name).unlink(missing_ok=True)
print("Saved model.joblib and metrics.json. Next: 03_scoring.ipynb.")

## Explain the result honestly

A strong result here supports an offline ranking experiment on this dataset. It does not demonstrate
future performance, fairness across groups, campaign savings, or production readiness. The dataset is
small and anonymous. A grouped random holdout is not a chronological backtest or a company deployment.